<a href="https://colab.research.google.com/github/junaidkhan035/Covid-19-x-ray-detectiion-Cnn/blob/main/Predictive_Maintenance_for_Industrial_Equipment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Step 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 2: Load the dataset
df = pd.read_csv('/content/data.csv')

In [3]:
df.head()

,time,V_avg_machine,I_avg_machine,kW_machine,kvar_machine,kVA_machine,PF_machine,kWh_machine,kvarh_machine,kVAh_machine,...,I_avg_spindle,kW_spindle,kvar_spindle,kVA_spindle,PF_spindle,kWh_spindle,kvarh_spindle,kVAh_spindle,RPM,Anomaly
0,2023-05-04 21:06:25,222.555862,3.978290,1.184035,0.968490,1.529677,0.774042,1.816337,1.503314,2.358233,...,5.533978,0.152449,2.475371,2.480061,0.061470,0.164464,1.580153,1.598300,2500,False
1,2023-05-04 21:06:26,222.595947,3.980162,1.184236,0.968880,1.530079,0.773970,1.816667,1.503583,2.358658,...,5.545718,0.152649,2.477063,2.481762,0.061508,0.164549,1.581529,1.599683,2500,False
2,2023-05-04 21:06:27,222.631027,3.984233,1.184538,0.968671,1.530181,0.774116,1.816996,1.503852,2.359083,...,5.550326,0.152649,2.478667,2.483363,0.061469,0.164591,1.582217,1.600376,2500,False
3,2023-05-04 21:06:28,222.631027,3.984233,1.184538,0.968671,1.530181,0.774116,1.816996,1.503852,2.359083,...,5.550326,0.152649,2.478667,2.483363,0.061469,0.164591,1.582217,1.600376,2500,False
4,2023-05-04 21:06:29,222.526733,3.982668,1.184841,0.970051,1.531290,0.773754,1.817654,1.504391,2.359936,...,5.559220,0.152848,2.482263,2.486965,0.061460,0.164677,1.583596,1.601762,2500,False


In [4]:
print("Shape:", df.shape)

Shape: (1214, 21)


In [5]:
# Step 3: Data exploration
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1214 entries, 0 to 1213
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   time           1214 non-null   object 
 1   V_avg_machine  1214 non-null   float64
 2   I_avg_machine  1214 non-null   float64
 3   kW_machine     1214 non-null   float64
 4   kvar_machine   1214 non-null   float64
 5   kVA_machine    1214 non-null   float64
 6   PF_machine     1214 non-null   float64
 7   kWh_machine    1214 non-null   float64
 8   kvarh_machine  1214 non-null   float64
 9   kVAh_machine   1214 non-null   float64
 10  V_avg_spindle  1214 non-null   float64
 11  I_avg_spindle  1214 non-null   float64
 12  kW_spindle     1214 non-null   float64
 13  kvar_spindle   1214 non-null   float64
 14  kVA_spindle    1214 non-null   float64
 15  PF_spindle     1214 non-null   float64
 16  kWh_spindle    1214 non-null   float64
 17  kvarh_spindle  1214 non-null   float64
 18  kVAh_spi

In [6]:
print(df.describe())

       V_avg_machine  I_avg_machine   kW_machine  kvar_machine  kVA_machine  \
count    1214.000000    1214.000000  1214.000000   1214.000000  1214.000000   
mean      221.130681       4.363567     1.288541      1.052339     1.668490   
std         1.487958       2.941508     0.511415      0.337606     0.599506   
min       217.005981       3.675694     0.536546      0.889565     1.403434   
25%       219.974369       3.798227     1.122918      0.914633     1.446863   
50%       220.365982       3.962649     1.180614      0.974336     1.530187   
75%       222.871056       4.211515     1.260790      1.025655     1.625529   
max       225.106888      40.942142     7.056128      3.989811     7.652349   

        PF_machine  kWh_machine  kvarh_machine  kVAh_machine  V_avg_spindle  \
count  1214.000000  1214.000000    1214.000000   1214.000000    1214.000000   
mean      0.770117    10.504549       8.868777     13.749598     270.197580   
std       0.042973     6.854458       5.817179     

In [8]:
print(df.isnull().sum())

time             0
V_avg_machine    0
I_avg_machine    0
kW_machine       0
kvar_machine     0
kVA_machine      0
PF_machine       0
kWh_machine      0
kvarh_machine    0
kVAh_machine     0
V_avg_spindle    0
I_avg_spindle    0
kW_spindle       0
kvar_spindle     0
kVA_spindle      0
PF_spindle       0
kWh_spindle      0
kvarh_spindle    0
kVAh_spindle     0
RPM              0
Anomaly          0
dtype: int64


In [14]:
# Step 4: Drop time column if not used directly
df.drop(columns=['time'], inplace=True)

In [15]:
# Step 5: Define features and target
X = df.drop(columns=['Anomaly'])
y = df['Anomaly']

In [16]:
# Step 6: Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [17]:
# Step 7: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [18]:
# Step 8: Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [19]:
# Step 9: Predictions and evaluation
y_pred = model.predict(X_test)

In [21]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Confusion Matrix:
 [[242   0]
 [  1   0]]


In [22]:
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

       False       1.00      1.00      1.00       242
        True       0.00      0.00      0.00         1

    accuracy                           1.00       243
   macro avg       0.50      0.50      0.50       243
weighted avg       0.99      1.00      0.99       243



In [23]:
# Step 10: Cross-validation
scores = cross_val_score(model, X_scaled, y, cv=5)
print("Cross-validation scores:", scores)
print("Mean accuracy:", np.mean(scores))

Cross-validation scores: [1.         1.         1.         0.99588477 0.07438017]
Mean accuracy: 0.8140529877903615
